# Tools

## Goal

Understanding:

- What a LangChain tool actually is, not just how to write `@tool`.
- Why does an LLM need tools.
- What makes a python function a LangChain tool?
- How does the LLM know which tools exist?
- Who decides when a tool should be called?
- Who actually executes the tool?
- How does the tool result get back into the LLM conversation?
- What is the difference between a Tool and simply calling a python function?
- How does this become the foundation for agents and LangGraph?

In [1]:
# Load the environment
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
# Read the model name
import os
MODEL_NAME = os.environ["GEMINI_MODEL"]
API_KEY = os.environ["GOOGLE_GENERATIVE_AI_API_KEY"]

In [3]:
# Create the LangChain model
from langchain_google_genai import ChatGoogleGenerativeAI
llm = ChatGoogleGenerativeAI (model = MODEL_NAME, api_key =API_KEY, temperature = 0)

## Get Weather tool

In [4]:
# Creating tool
from langchain.tools import tool

@tool
def get_weather(city: str):
    """Get the current weather for a city. """
    return f"The weather in {city} is sunny. "


**Conceptually**

```text
                LangChain Tool
                     │
        ┌────────────┼────────────┐
        ↓            ↓            ↓
      Name       Description    Schema
        │            │            │
 get_weather   Get current...   city: str
                     │
                     ↓
              Python function
```

In [5]:
print(get_weather.name)
print(get_weather.description)
print(get_weather.args)

get_weather
Get the current weather for a city.
{'city': {'title': 'City', 'type': 'string'}}


In [6]:
# Creates tool object.
llm_with_tools = llm.bind_tools([get_weather]) 

1. `bind_tools()`: Tell the model what tools exist
2. `invoke()`: Ast the model what to do
3. tool execution: Actually run the python function.

In [7]:
# Invoke the tool
response = llm_with_tools.invoke( # Gemini call
    "What's the weather in Cairo"
)

In [8]:
print(response,"\n")
print(response.tool_calls)

content=[] additional_kwargs={'function_call': {'name': 'get_weather', 'arguments': '{"city": "Cairo"}'}, '__gemini_function_call_thought_signatures__': {'call_1384706': 'EvQCCvECARFNMg9W6ISMr857lQtMLXec75WNps2mPITm4avPjV9zuq0kbj/av6D8eIP1SJ2hIbpzs7UBl+/dE86U8Mph/7KAjfy7paqAHeBOCqbWl+t0HmjLVLWcqoqqbT6YIHhn9bXH9OIWNdWBfMMwFco3IlMMDdzP949yXrEzZ9jsgLZy0Niuf6xCOOgtOK1JKqIUhGV1Epd6nL130lJl/i5mUTKFyK7TXBGfc5BjA0qhih90YVhr3GC3r10mBiMRcbAZjWOZAhO7af2cv7LMFlmUliKxivPgSdAlScBbLxaee7LwLcHYyzmFYzxTBDLx3ebj2oPi914bPvOLP9qo50hBkbf3pzHGa4cJ+ykXHW2K7ORDdHHUeY4jqgBn7xqf/jMUJMt3CsWNHFcEJ16DK1/iO1epY3GF2ElUKqOooh9JZeW/3/xRVo/nsfT6A9llVmlwoq0Hmu89JqzQWbmL23QG0XiWxNGGHxWtVeo7ACztbHVS'}} response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'} id='lc_run--01a02e82-d535-70b0-ab5f-b9b36fa77211-0' tool_calls=[{'name': 'get_weather', 'args': {'city': 'Cairo'}, 'id': 'call_1384706', 'type': 'tool_call'}] invalid_tool_calls=[] usage_meta

In [9]:
response.tool_calls

[{'name': 'get_weather',
  'args': {'city': 'Cairo'},
  'id': 'call_1384706',
  'type': 'tool_call'}]

In [10]:
tool_call  = response.tool_calls[0] 
tool_call 

{'name': 'get_weather',
 'args': {'city': 'Cairo'},
 'id': 'call_1384706',
 'type': 'tool_call'}

In [11]:
result = get_weather.invoke(tool_call['args'])
print(result)

The weather in Cairo is sunny. 


```text
Gemini
  │
  │ tool call
  ↓
AIMessage
  │
  │ response.tool_calls
  ↓
Application
  │
  │ get_weather.invoke(args)
  ↓
Tool
  │
  ↓
"the weather in Cairo is sunny"
```

## Tool message

sequence:

```text
HumanMessage
      ↓
AIMessage
      ↓
ToolMessage
      ↓
Gemini
```

In [12]:
from langchain_core.messages import ToolMessage

tool_call = response.tool_calls[0]

result = get_weather.invoke(tool_call["args"])

tool_message = ToolMessage(
    content=result,
    tool_call_id = tool_call["id"]
)

print(tool_message)

content='The weather in Cairo is sunny. ' tool_call_id='call_1384706'


In [13]:
# Build the human message.
from langchain_core.messages import HumanMessage

messages = [
    HumanMessage(content = "What's the weather in Cairo?"),
    response,
    tool_message,
]

```text
messages
│
├── HumanMessage
│      "What's the weather in Cairo?"
│
├── AIMessage
│      tool_calls:
│          get_weather(city="Cairo")
│
└── ToolMessage
       tool_call_id:
           call_1635569
       content:
           "The weather in Cairo is sunny."
```

In [14]:
final_response = llm_with_tools.invoke(messages)

In [15]:
print(final_response)
print(final_response.content)

content=[{'type': 'text', 'text': 'The weather in Cairo is currently sunny.', 'extras': {'signature': 'EmgKZgERTTIPi3F3DJQk392hNW6gqrWlxN3nhcjEP43+QQ4RE6Ix++ZRf7idmmyzR0cTFwE/NzHfGrdm8HF4NgJdiQTuIqGEkiRkbERR8mpkBVpMnCEp6DNhcP45K7u57PeFLKIUQOJtNg=='}}] additional_kwargs={} response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'} id='lc_run--01a02e82-ef86-7581-803e-19c168e48411-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 153, 'output_tokens': 8, 'total_tokens': 161, 'input_token_details': {'cache_read': 0}}
[{'type': 'text', 'text': 'The weather in Cairo is currently sunny.', 'extras': {'signature': 'EmgKZgERTTIPi3F3DJQk392hNW6gqrWlxN3nhcjEP43+QQ4RE6Ix++ZRf7idmmyzR0cTFwE/NzHfGrdm8HF4NgJdiQTuIqGEkiRkbERR8mpkBVpMnCEp6DNhcP45K7u57PeFLKIUQOJtNg=='}}]


Workflow:

```text
User
 │
 │ "What's the weather in Cairo?"
 ↓
llm_with_tools.invoke()
 │
 ↓
Gemini
 │
 │ AIMessage
 │ tool_calls = get_weather(Cairo)
 ↓
Application
 │
 │ get_weather.invoke({"city": "Cairo"})
 ↓
Tool
 │
 │ "The weather in Cairo is sunny."
 ↓
ToolMessage
 │
 │ tool_call_id = ...
 ↓
llm_with_tools.invoke(messages)
 │
 ↓
Gemini
 │
 │ AIMessage
 │ tool_calls = []
 │ content = "The weather in Cairo is currently sunny."
 ↓
User
```